# Multi-source batch (APTOS + DDR training): the no-VQC model, seeds 42, 43, 44, unattended

Same design and settings as the multi-source VQC runs (20 epochs, tag `ms`, checkpoint = best mean of the APTOS-val and DDR-val QWKs, IDRiD never used for any decision). Only `quantum_mode` differs: `"skip"` = the eight values go straight to the fusion head, with no circuit.

For each run it trains, then tests once on APTOS test, DDR held-out test and IDRiD, and saves files under the same naming scheme as the manual runs (`ms_skip_aNone_pe0_L4_s42`, ...).

**Run cells top to bottom; then wait.** If the session breaks, run the same cells again with the same settings: finished runs are skipped and an unfinished run resumes.
1. Fill the three zip paths in cell 1.
2. Leave `SMOKE = True` for the first pass (2 epochs of the first run only). Check there are no errors.
3. Set `SMOKE = False` in cell 2, then run cell 2 and everything below it.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile

APTOS_ZIP = "/content/drive/MyDrive/your folder.zip"
IDRID_ZIP = "/content/drive/MyDrive/your folder.zip"        # <-- CHANGE
DDR_ZIP   = "/content/drive/MyDrive/your folder.zip"          # <-- CHANGE

os.system("pip install -q timm pennylane pennylane-lightning")
if not os.path.exists("/content/APTOS"):
    zipfile.ZipFile(APTOS_ZIP).extractall("/content/APTOS")
if not os.path.exists("/content/idrid/IDRID"):
    os.system(f'unzip -q "{IDRID_ZIP}" -d /content/idrid')
if not os.path.exists("/content/ddr/DR_grading.csv"):
    os.system(f'unzip -q "{DDR_ZIP}" -d /content/ddr')
print(os.listdir("/content/APTOS"), os.listdir("/content/idrid"), os.listdir("/content/ddr"))

Mounted at /content/drive
['test_images', 'val_images', 'valid.csv', 'train_1.csv', 'test.csv', 'train_images'] ['IDRID'] ['DR_grading.csv', 'DR_grading']


## 2. Imports and the list of runs

In [ ]:
import random, hashlib, time
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import timm, cv2
import pennylane as qml
from concurrent.futures import ThreadPoolExecutor
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score, roc_auc_score,
                             classification_report, confusion_matrix)

BASE_PATH = "/content/drive/MyDrive/your folder"
os.makedirs(BASE_PATH + "/checkpoints", exist_ok=True)

SMOKE = False        # True = 2-epoch test of the first run only. Set False for the real batch.

RUN_LIST = [("skip", 42), ("skip", 43), ("skip", 44)]       # (quantum_mode, seed)

config = {"epochs": 20, "batch_size": 32, "lr_backbone": 1e-4, "lr_other": 1e-4, "lr_alpha": 1e-2,
          "weight_decay": 0.01, "n_trans_layers": 4, "tag": "ms"}
if SMOKE:
    config["epochs"], config["tag"], RUN_LIST = 2, "mssmokebatch", RUN_LIST[:1]
print("SMOKE" if SMOKE else "REAL BATCH", "| epochs", config["epochs"], "| tag", config["tag"], "|", RUN_LIST)

REAL BATCH | epochs 20 | tag ms | [('skip', 42), ('skip', 43), ('skip', 44)]


## 3. Augmentation (same as every other experiment)

In [ ]:
train_tf = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop(size=(224, 224), scale=(0.85, 1.0)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Rotate(limit=15),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## 4. Data: APTOS (cleaned) + DDR (cleaned), IDRiD for the final test, each cached once at 256x256

In [ ]:
def cache_one(a):
    s, d = a
    if not os.path.exists(d):
        im = cv2.imread(s)
        if im is None: raise RuntimeError("cannot read " + s)
        cv2.imwrite(d, cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA))
def cache_all(df):
    with ThreadPoolExecutor(4) as ex:
        list(tqdm(ex.map(cache_one, zip(df.src_path, df.cache)), total=len(df)))

APT_ROOT = "/content/APTOS"
ap_dir = {"train": f"{APT_ROOT}/train_images/train_images", "val": f"{APT_ROOT}/val_images/val_images",
          "test": f"{APT_ROOT}/test_images/test_images"}
ap = pd.read_csv(BASE_PATH + "/aptos_clean_split.csv")[["id_code", "diagnosis", "split"]].copy()
ap["source"] = "aptos"
ap["src_path"] = [f"{ap_dir[s]}/{i}.png" for s, i in zip(ap.split, ap.id_code)]
ap["cache"] = [f"/content/cache_aptos/{i}.png" for i in ap.id_code]

by_name = {}
for root, _, files in os.walk("/content/ddr"):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")): by_name.setdefault(f, []).append(os.path.join(root, f))
dd = pd.read_csv(BASE_PATH + "/ddr_clean_split.csv")[["id_code", "diagnosis", "split"]].copy()
dd["source"] = "ddr"
missing = [n for n in dd.id_code if n not in by_name]
assert not missing, f"{len(missing)} DDR images not found, e.g. {missing[:3]}"
dd["src_path"] = dd.id_code.map(lambda n: by_name[n][0])
dd["cache"] = [f"/content/cache_ddr/{i}.png" for i in dd.id_code]

alldf = pd.concat([ap, dd], ignore_index=True)
alldf["diagnosis"] = alldf["diagnosis"].astype(int)
for d in ("/content/cache_aptos", "/content/cache_ddr"): os.makedirs(d, exist_ok=True)
cache_all(alldf)
for (s, sp), gdf in alldf.groupby(["source", "split"]):
    print(f"{s:6s} {sp:5s} {len(gdf):6d}", np.bincount(gdf.diagnosis, minlength=5))

# IDRiD: all 516 graded images (used only in the final test of each run)
R = "/content/idrid/IDRID"
img_dirs  = {"train": f"{R}/1. Original Images/a. Training Set", "test": f"{R}/1. Original Images/b. Testing Set"}
lab_files = {"train": f"{R}/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv",
             "test":  f"{R}/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv"}
rows = []
for sub in ("train", "test"):
    lab = pd.read_csv(lab_files[sub]).iloc[:, :2].dropna(); lab.columns = ["name", "grade"]
    files = {os.path.splitext(f)[0]: os.path.join(img_dirs[sub], f) for f in os.listdir(img_dirs[sub])
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))}
    for _, r in lab.iterrows():
        n = str(r["name"]).strip()
        if n in files: rows.append(dict(sub=sub, name=n, src_path=files[n], diagnosis=int(r["grade"])))
idr = pd.DataFrame(rows)
idr["cache"] = [f"/content/cache_idrid/{s}_{n}.png" for s, n in zip(idr["sub"], idr["name"])]
os.makedirs("/content/cache_idrid", exist_ok=True); cache_all(idr)
print("IDRiD", len(idr), np.bincount(idr.diagnosis, minlength=5))

class CachedDS(Dataset):
    def __init__(self, df, tf):
        self.p, self.y, self.t = df.cache.tolist(), df.diagnosis.astype(int).tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = cv2.cvtColor(cv2.imread(self.p[i]), cv2.COLOR_BGR2RGB)
        return self.t(image=im)["image"], self.y[i]

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

bs = config["batch_size"]
train_df = alldf[alldf.split == "train"]
mk = lambda df: DataLoader(CachedDS(df, val_tf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
val_loaders  = {s: mk(alldf[(alldf.split == "val")  & (alldf.source == s)]) for s in ("aptos", "ddr")}
test_loaders = {s: mk(alldf[(alldf.split == "test") & (alldf.source == s)]) for s in ("aptos", "ddr")}
idrid_loader = mk(idr)
print("training images:", len(train_df))

100%|██████████| 15928/15928 [08:10<00:00, 32.47it/s] 


aptos  test     339 [197  28  75  12  27]
aptos  train   2824 [1427  277  756  145  219]
aptos  val      341 [172  33  91  20  25]
ddr    test    1864 [926  94 672  35 137]
ddr    train   8696 [4317  441 3134  165  639]
ddr    val     1864 [925  95 671  36 137]


100%|██████████| 516/516 [00:46<00:00, 11.02it/s]


IDRiD 516 [168  25 168  93  62]
training images: 11520


## 5. Quantum circuit and model (as in the multi-source VQC runs; classical simulator, backpropagation)

In [ ]:
n_qubits = 8
n_layers = 3

dev = qml.device("default.qubit", wires=n_qubits)

def quantum_circuit(inputs, weights):
    for i in range(n_qubits):                       # initial U3 encoding
        qml.U3(inputs[..., i], inputs[..., i], inputs[..., i], wires=i)
    for l in range(n_layers):
        for i in range(n_qubits):                   # data re-uploading
            qml.RY(inputs[..., i], wires=i)
        for i in range(n_qubits):                   # trainable single-qubit rotations (72 parameters in total)
            qml.U3(weights[l][i][0], weights[l][i][1], weights[l][i][2], wires=i)
        for i in range(n_qubits):                   # ring entanglement
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qnode = qml.QNode(quantum_circuit, dev, interface="torch", diff_method="backprop")

In [ ]:
class QuantumBranch(nn.Module):
    """256 -> attention gate -> reducer -> 8 angles -> (circuit | control) -> 8 features"""
    def __init__(self, feature_dim=256, mode="circuit"):
        super().__init__()
        self.mode = mode

        self.attn = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.ReLU(), nn.Linear(64, feature_dim)
        )
        self.reducer = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),          nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, n_qubits)
        )

        if mode in ("circuit", "frozen"):
            self.quantum = qml.qnn.TorchLayer(qnode, weight_shapes)   # a fresh circuit for every model
            if mode == "frozen":                                      # random circuit that is never trained
                for p in self.quantum.parameters():
                    p.requires_grad_(False)
        elif mode == "mlp":                                           # matched classical layer: 8*8 + 8 = 72 parameters
            self.quantum = nn.Sequential(nn.Linear(n_qubits, n_qubits), nn.Tanh())
        elif mode == "skip":                                          # no circuit at all
            self.quantum = None
        else:
            raise ValueError(f"unknown quantum_mode: {mode}")

    def forward(self, x):
        a = torch.softmax(self.attn(x), dim=-1)
        x = x * a
        x = self.reducer(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = torch.tanh(x) * np.pi                       # 8 angles in (-pi, pi)

        if self.mode in ("circuit", "frozen"):
            q_out = self.quantum(x.float())
        elif self.mode == "mlp":
            q_out = self.quantum(x.float() / np.pi)
        else:                                            # "skip": same 8 numbers, scaled to (-1, 1) like the circuit output
            q_out = x.float() / np.pi
        return q_out, x

In [ ]:
class HybridMSLATNet(nn.Module):
    def __init__(self, num_classes=5, embed_dim=256, n_trans_layers=4, use_pos_emb=False,
                 quantum_mode="circuit", fixed_alpha=None):
        super().__init__()

        self.backbone = timm.create_model("efficientnet_b2", pretrained=True, features_only=True)
        feat_dims = [f["num_chs"] for f in self.backbone.feature_info[-3:]]

        self.proj = nn.ModuleList([nn.Linear(feat_dims[i], embed_dim) for i in range(3)])

        # optional learned position / scale embeddings (one vector per token of the concatenated sequence)
        self.pos_emb = None
        if use_pos_emb:
            was_training = self.backbone.training
            self.backbone.eval()
            with torch.no_grad():
                fs = self.backbone(torch.zeros(1, 3, 224, 224))[-3:]
            n_tokens = sum(f.shape[2] * f.shape[3] for f in fs)
            self.backbone.train(was_training)
            self.pos_emb = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
            nn.init.trunc_normal_(self.pos_emb, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=embed_dim * 4,
            dropout=0.5, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_trans_layers, enable_nested_tensor=False)

        self.pool = nn.Linear(embed_dim, 1)

        self.classical_head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

        self.quantum_branch = QuantumBranch(embed_dim, mode=quantum_mode)

        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim + n_qubits, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.quantum_head = nn.Sequential(
            nn.Linear(n_qubits, 32), nn.GELU(), nn.Linear(32, num_classes)
        )

        # mixing weight between the classical and the fused prediction
        self.learn_alpha = fixed_alpha is None
        if self.learn_alpha:
            self.alpha = nn.Parameter(torch.tensor(-1.0))               # sigmoid(-1) = 0.269 at the start
        else:
            self.register_buffer("alpha", torch.tensor(float(fixed_alpha)))

    def get_alpha(self):
        return torch.sigmoid(self.alpha) if self.learn_alpha else self.alpha

    def forward(self, x):
        feats = self.backbone(x)[-3:]

        tokens = []
        for i, f in enumerate(feats):
            t = f.flatten(2).transpose(1, 2)          # (B, H*W, C): every spatial position is one token
            tokens.append(self.proj[i](t))
        tokens = torch.cat(tokens, dim=1)
        if self.pos_emb is not None:
            tokens = tokens + self.pos_emb
        tokens = self.transformer(tokens)

        attn = torch.softmax(self.pool(tokens), dim=1)
        pooled = (tokens * attn).sum(dim=1)            # (B, 256)

        classical_logits = self.classical_head(pooled)

        quantum_features, q_input = self.quantum_branch(pooled)
        fusion_logits  = self.fusion_head(torch.cat([pooled, quantum_features], dim=1))
        quantum_logits = self.quantum_head(quantum_features)

        alpha = self.get_alpha()
        out = (1 - alpha) * classical_logits + alpha * fusion_logits

        return out, classical_logits, quantum_logits, pooled, attn, q_input

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, smoothing=0.05):
        super().__init__()
        self.gamma, self.smoothing = gamma, smoothing

    def forward(self, logits, targets, reduction="mean"):
        logits = logits.float()
        K = logits.size(1)
        with torch.no_grad():
            t = torch.full_like(logits, self.smoothing / (K - 1))
            t.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        ce = -(t * F.log_softmax(logits, dim=1)).sum(dim=1)
        loss = (1 - torch.exp(-ce)) ** self.gamma * ce
        return loss if reduction == "none" else loss.mean()

def ordinal_loss(logits, targets):
    # P(grade <= j) must be 0 for j < true grade and 1 for j >= true grade
    cum = torch.cumsum(torch.softmax(logits.float(), dim=1), dim=1)
    idx = torch.arange(cum.size(1), device=cum.device).unsqueeze(0)
    tgt = (idx >= targets.unsqueeze(1)).float()
    return F.mse_loss(cum, tgt)

def compute_total_loss(hybrid_logits, classical_logits, quantum_logits, targets, focal_fn):
    w = torch.ones_like(targets, dtype=torch.float32)
    w[targets >= 3] = 1.5                                            # per-sample severity weight
    hybrid_loss    = (w * focal_fn(hybrid_logits, targets, "none")).mean()
    classical_loss = focal_fn(classical_logits, targets)
    quantum_loss   = focal_fn(quantum_logits, targets)
    ord_loss       = ordinal_loss(hybrid_logits, targets)
    return hybrid_loss + 0.2 * quantum_loss + 0.3 * classical_loss + 0.1 * ord_loss

## 6. The batch

In [ ]:
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
model = None
names = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]
res_path = f"{BASE_PATH}/results_multisource.csv"
def log_row(**row):
    pd.DataFrame([row]).to_csv(res_path, mode="a", header=not os.path.exists(res_path), index=False)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def evaluate(loader):
    model.eval(); P, Y = [], []
    for imgs, labels in loader:
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(imgs.to(device, non_blocking=True))[0]
        P.append(torch.softmax(out.float(), 1).cpu().numpy()); Y.append(labels.numpy())
    P = np.concatenate(P).astype(np.float64); P = P / P.sum(1, keepdims=True)
    Y = np.concatenate(Y); pred = P.argmax(1)
    L = [0, 1, 2, 3, 4]; cm = confusion_matrix(Y, pred, labels=L)
    m = dict(acc=accuracy_score(Y, pred),
             f1=f1_score(Y, pred, average="macro", labels=L, zero_division=0),
             qwk=cohen_kappa_score(Y, pred, weights="quadratic", labels=L),
             auc=roc_auc_score(Y, P, multi_class="ovr", labels=L),
             ref_auc=roc_auc_score((Y >= 2).astype(int), P[:, 2:].sum(1)),
             sens=cm[2:, 2:].sum() / cm[2:].sum(), spec=cm[:2, :2].sum() / cm[:2].sum())
    return m, P, Y, pred

def run_one(mode, seed):
    global model
    RUN = f"{config['tag']}_{mode}_aNone_pe0_L{config['n_trans_layers']}_s{seed}"
    done_marker = f"{BASE_PATH}/ext_idrid_{RUN}.npz"
    if os.path.exists(done_marker):
        print("\nSKIP (already finished):", RUN); return
    t_start = time.time()
    print("\n" + "=" * 90 + f"\nRUN {RUN}\n" + "=" * 90)

    set_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(CachedDS(train_df, train_tf), batch_size=bs, shuffle=True, num_workers=2, pin_memory=True,
                              generator=g, worker_init_fn=seed_worker, drop_last=True)
    model = HybridMSLATNet(n_trans_layers=config["n_trans_layers"], use_pos_emb=False, quantum_mode=mode, fixed_alpha=None).to(device)
    criterion = FocalLoss(gamma=2, smoothing=0.05)
    bb = list(model.backbone.parameters()); bb_ids = {id(p) for p in bb}
    alpha_params = [model.alpha]; al_ids = {id(p) for p in alpha_params}
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in bb_ids and id(p) not in al_ids]
    optimizer = torch.optim.AdamW([{"params": bb, "lr": config["lr_backbone"]},
                                   {"params": other, "lr": config["lr_other"]},
                                   {"params": alpha_params, "lr": config["lr_alpha"], "weight_decay": 0.0}],
                                  weight_decay=config["weight_decay"])
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, config["epochs"] - 5))
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])
    print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M | schedule: {config['epochs']} epochs, cosine T_max {cosine.T_max}")

    best_path = f"{BASE_PATH}/checkpoints/best_{RUN}.pth"
    last_path = f"{BASE_PATH}/checkpoints/last_{RUN}.pth"
    start_epoch, best_sel, log = 0, -1.0, []
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["opt"]); scheduler.load_state_dict(ck["sched"])
        start_epoch, best_sel, log = ck["epoch"] + 1, ck["best_sel"], ck["log"]
        print(f"Resuming from epoch {start_epoch}, best mean val QWK {best_sel:.4f}")

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    for epoch in range(start_epoch, config["epochs"]):
        t0 = time.time(); model.train(); run_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out, cl, ql = model(imgs)[:3]
                loss = compute_total_loss(out, cl, ql, labels, criterion)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            run_loss += loss.item()
        scheduler.step()
        vm = {s: evaluate(l)[0] for s, l in val_loaders.items()}
        sel = float(np.mean([vm[s]["qwk"] for s in vm]))
        alpha = float(model.get_alpha().detach())
        row = dict(epoch=epoch + 1, train_loss=run_loss / len(train_loader), alpha=alpha, sel_qwk=sel, minutes=(time.time() - t0) / 60)
        for s in vm: row.update({f"{s}_{k}": v for k, v in vm[s].items()})
        log.append(row)
        print(f"Ep {epoch+1:03d} ({row['minutes']:.1f} min) loss {row['train_loss']:.4f} | mean val QWK {sel:.4f} | "
              f"APTOS QWK {vm['aptos']['qwk']:.4f} | DDR QWK {vm['ddr']['qwk']:.4f} | alpha {alpha:.3f}")
        if sel > best_sel:
            best_sel = sel
            torch.save({"epoch": epoch, "model": model.state_dict(), "alpha": alpha, "val": vm}, best_path)
        torch.save({"epoch": epoch, "model": model.state_dict(), "opt": optimizer.state_dict(),
                    "sched": scheduler.state_dict(), "best_sel": best_sel, "log": log}, last_path)
    pd.DataFrame(log).to_csv(f"{BASE_PATH}/train_log_{RUN}.csv", index=False)
    print("Training finished. Best mean validation QWK:", round(best_sel, 4))

    ck = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); be = ck["epoch"] + 1
    for s, l in test_loaders.items():
        tm, P, Y, pred = evaluate(l)
        print(f"\n--- {s.upper()} test ({len(Y)}), best epoch {be}, alpha {ck['alpha']:.3f} ---")
        print({k: round(float(v), 4) for k, v in tm.items()})
        print(classification_report(Y, pred, target_names=names, digits=4, zero_division=0))
        print(confusion_matrix(Y, pred, labels=[0, 1, 2, 3, 4]))
        np.savez(f"{BASE_PATH}/test_preds_{RUN}_{s}.npz", P=P, Y=Y)
        log_row(run=RUN, dataset=f"{s}_test", best_epoch=be, alpha=ck["alpha"], n=len(Y), **{k: float(v) for k, v in tm.items()})
    tm, P, Y, pred = evaluate(idrid_loader)
    print(f"\n--- IDRiD external ({len(Y)}) ---"); print({k: round(float(v), 4) for k, v in tm.items()})
    print(confusion_matrix(Y, pred, labels=[0, 1, 2, 3, 4]))
    log_row(run=RUN, dataset="idrid_516", best_epoch=be, alpha=ck["alpha"], n=len(Y), **{k: float(v) for k, v in tm.items()})
    np.savez(done_marker, P=P, Y=Y)                         # written last = marks the run as finished
    print(f"\nFINISHED {RUN} in {(time.time()-t_start)/60:.1f} min")
    del model; model = None
    if device.type == "cuda": torch.cuda.empty_cache()

for mode, seed in RUN_LIST:
    run_one(mode, seed)
print("\nALL RUNS IN THE LIST ARE FINISHED")


SKIP (already finished): ms_skip_aNone_pe0_L4_s42

SKIP (already finished): ms_skip_aNone_pe0_L4_s43

RUN ms_skip_aNone_pe0_L4_s44


model.safetensors: reconstructing file:   0%|          |  0.00B / 36.8MB            

model.safetensors: downloading bytes:           |  0.00B            

parameters: 10.75 M | schedule: 20 epochs, cosine T_max 15
Ep 001 (2.4 min) loss 1.2968 | mean val QWK 0.5537 | APTOS QWK 0.7116 | DDR QWK 0.3958 | alpha 0.351
Ep 002 (2.3 min) loss 0.9140 | mean val QWK 0.7666 | APTOS QWK 0.7872 | DDR QWK 0.7460 | alpha 0.480
Ep 003 (2.3 min) loss 0.7615 | mean val QWK 0.8229 | APTOS QWK 0.8629 | DDR QWK 0.7829 | alpha 0.570
Ep 004 (2.4 min) loss 0.6582 | mean val QWK 0.8467 | APTOS QWK 0.8876 | DDR QWK 0.8059 | alpha 0.656
Ep 005 (2.3 min) loss 0.5742 | mean val QWK 0.8379 | APTOS QWK 0.8723 | DDR QWK 0.8036 | alpha 0.722
Ep 006 (2.3 min) loss 0.5154 | mean val QWK 0.8608 | APTOS QWK 0.8822 | DDR QWK 0.8393 | alpha 0.768
Ep 007 (2.3 min) loss 0.4653 | mean val QWK 0.8668 | APTOS QWK 0.8941 | DDR QWK 0.8396 | alpha 0.809
Ep 008 (2.3 min) loss 0.4171 | mean val QWK 0.8739 | APTOS QWK 0.9008 | DDR QWK 0.8469 | alpha 0.829
Ep 009 (2.3 min) loss 0.3638 | mean val QWK 0.8793 | APTOS QWK 0.9105 | DDR QWK 0.8481 | alpha 0.845
Ep 010 (2.3 min) loss 0.3321 | m

## 7. Summary of the finished runs

In [ ]:
r = pd.read_csv(res_path)
r = r[~r.run.str.startswith("mssmoke")]
print(r[["run", "dataset", "best_epoch", "n", "acc", "f1", "qwk", "auc", "ref_auc", "sens", "spec"]].round(4).to_string(index=False))

                        run    dataset  best_epoch         n    acc     f1    qwk    auc  ref_auc   sens   spec
ms_circuit_aNone_pe0_L4_s42 aptos_test          19  339.0000 0.8555 0.6557 0.8921 0.9242   0.9747 0.9386 0.9422
ms_circuit_aNone_pe0_L4_s42   ddr_test          19 1864.0000 0.8230 0.6391 0.8471 0.9036   0.9497 0.8578 0.9235
ms_circuit_aNone_pe0_L4_s42  idrid_516          19    0.5097 0.4452 0.6889 0.8113 0.9272   0.9071 0.7150    NaN
   ms_skip_aNone_pe0_L4_s42 aptos_test          18  339.0000 0.8466 0.6155 0.9050 0.9022   0.9735 0.9474 0.9200
   ms_skip_aNone_pe0_L4_s42   ddr_test          18 1864.0000 0.8197 0.5970 0.8415 0.9055   0.9563 0.8791 0.8922
   ms_skip_aNone_pe0_L4_s42  idrid_516          18  516.0000 0.5116 0.4341 0.6398 0.8248   0.9205 0.9257 0.5959
   ms_skip_aNone_pe0_L4_s43 aptos_test          13  339.0000 0.8319 0.6101 0.9026 0.9443   0.9819 0.9386 0.9244
   ms_skip_aNone_pe0_L4_s43   ddr_test          13 1864.0000 0.8369 0.5956 0.8639 0.9239   0.9616 0.9123